# Practice Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for every exercise in [1.3-numpy-practice-exercises.ipynb](1.3-numpy-practice-exercises.ipynb).
:::

## Exercise 1: Create and inspect

Build the following 3×4 field of temperatures (°C) as a numpy array:

`5.2 4.8 6.1 3.9 1.3 0.5 -0.8 -1.2 -2.6 -3.1 -1.9 0.2`

Print its `shape`, `ndim`, and `dtype`, and its overall mean rounded to two decimals.
Then create, and print, three arrays of the same shape without typing the shape out by hand:
all ones, a uniform reference field of 15.0 °C, and random values in [0, 1).

In [ ]:
import numpy as np

field_celsius = np.array([
    [ 5.2,  4.8,  6.1,  3.9],
    [ 1.3,  0.5, -0.8, -1.2],
    [-2.6, -3.1, -1.9,  0.2],
])
print(field_celsius.shape, field_celsius.ndim, field_celsius.dtype)
print(round(float(field_celsius.mean()), 2))

print(np.ones(field_celsius.shape))
print(np.full(field_celsius.shape, 15.0))       # reference field, °C
print(np.random.random(field_celsius.shape))

## Exercise 2: Index and slice

Given

```python
a = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
```

print the last row, the first column, and the 2×2 sub-block formed by the first two rows and the last two columns. 

Finally, take a slice of the first row, change its first element to `99.0`, and print the original array. Then repeat using `.copy()`. Explain the difference in a comment.

In [ ]:
import numpy as np

a = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
print(a[-1, :])
print(a[:, 0])
print(a[0:2, 2:4])

row = a[0, :]       # a slice is a view, not a copy
row[0] = 99.0
print(a)            # the original changed too

b = np.array([[1.0, 2.0, 3.0, 4.0],
              [5.0, 6.0, 7.0, 8.0],
              [9.0, 10.0, 11.0, 12.0]])
row_copy = b[0, :].copy()
row_copy[0] = 99.0
print(b)            # unaffected: .copy() breaks the shared memory

## Exercise 3: Masking and np.where

Given

```python
temp_celsius = np.array([[-1.0, 12.0,  3.0],
                         [ 8.0, -2.0, 15.0],
                         [ 4.0,  0.5, 11.0]])
```

print the values that exceed 10 °C, their mean, and the percentage of the field they make up.
Then build a clipped field in which every value below 0 °C is set to 0, using `np.where`.

In [ ]:
import numpy as np

temp_celsius = np.array([[-1.0, 12.0,  3.0],
                         [ 8.0, -2.0, 15.0],
                         [ 4.0,  0.5, 11.0]])
mask = temp_celsius > 10.0
print(temp_celsius[mask])
print(temp_celsius[mask].mean())
print(f"{100.0 * mask.sum() / temp_celsius.size:.1f} % of the field")

clipped = np.where(temp_celsius < 0.0, 0.0, temp_celsius)
print(clipped)

## Exercise 4: Broadcasting and grids

Given the field below, add a per-column offset (length 4) to every row, and separately add a per-row offset (length 3) to every column.

```python
field = np.array([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0],
                  [9.0, 10.0, 11.0, 12.0]])
col_offset = np.array([0.0, 1.0, 2.0, 3.0])
row_offset = np.array([10.0, 20.0, 30.0])
```

Then build a grid of 4 evenly spaced longitudes from 6 to 9 °E and 3 evenly spaced latitudes
from 46 to 47 °N, so that it has the same shape as `field`. On that grid, compute a synthetic temperature
field that is 15.0 °C at (6 °E, 46 °N), cools by 0.6 °C per degree of latitude northward, and warms
by 0.2 °C per degree of longitude eastward. Compute it twice, once from the 2D arrays that
`np.meshgrid` returns and once by broadcasting the 1D latitudes and longitudes directly, and
confirm with `np.allclose` that the two agree.

In [ ]:
import numpy as np

field = np.array([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0],
                  [9.0, 10.0, 11.0, 12.0]])
col_offset = np.array([0.0, 1.0, 2.0, 3.0])
row_offset = np.array([10.0, 20.0, 30.0])

print(field + col_offset)            # (3,4) + (4,) broadcasts across rows
print(field + row_offset[:, None])   # (3,4) + (3,1) broadcasts across columns

lon = np.linspace(6.0, 9.0, 4)     # 4 longitudes, °E
lat = np.linspace(46.0, 47.0, 3)   # 3 latitudes, °N

lon2d, lat2d = np.meshgrid(lon, lat)                              # both (3, 4)
temp_grid_celsius = 15.0 - 0.6 * (lat2d - 46.0) + 0.2 * (lon2d - 6.0)

# the same field without meshgrid: a (3, 1) column plus a (4,) row broadcasts to (3, 4)
temp_bcast_celsius = 15.0 - 0.6 * (lat[:, None] - 46.0) + 0.2 * (lon - 6.0)

print(temp_grid_celsius)
print(temp_grid_celsius.shape, temp_bcast_celsius.shape)         # (3, 4) (3, 4)
print(np.allclose(temp_grid_celsius, temp_bcast_celsius))        # True

## Exercise 5: Axis reductions

Given seasonal mean temperatures at three stations, one row per station and one column per season
(winter, spring, summer, autumn):

```python
seasonal_celsius = np.array([[ 2.1,  9.8, 18.4, 10.2],
                             [-1.5,  7.2, 16.9,  8.0],
                             [ 4.0, 11.5, 20.3, 12.6]])
```

print the annual mean at each station and the index of each station's warmest season. Then find the
coldest value in each season twice, once with the method and once with the numpy function, and
confirm they agree.

Finally, given four days of rainfall at two stations, one row per station,

```python
rain_mm = np.array([[0.0, 6.2, 1.1, 0.0],
                    [2.4, 0.0, 0.0, 8.3]])
```

use `cumsum` along the right axis to get the rain to date. Print the index of the station that was
wetter after the second day and the index of the station that was wetter after all four. Are they
the same station?

In [ ]:
import numpy as np

seasonal_celsius = np.array([[ 2.1,  9.8, 18.4, 10.2],
                             [-1.5,  7.2, 16.9,  8.0],
                             [ 4.0, 11.5, 20.3, 12.6]])

print(seasonal_celsius.mean(axis=1))      # annual mean at each station
print(seasonal_celsius.argmax(axis=1))    # [2 2 2]: summer at every station

print(seasonal_celsius.min(axis=0))       # coldest value in each season, as a method
print(np.min(seasonal_celsius, axis=0))   # the same, as a numpy function

rain_mm = np.array([[0.0, 6.2, 1.1, 0.0],
                    [2.4, 0.0, 0.0, 8.3]])
rain_to_date_mm = rain_mm.cumsum(axis=1)  # stations are rows, so accumulate along axis 1
print(rain_to_date_mm)
print(int(rain_to_date_mm[:, 1].argmax()))    # 0: wetter after day 2
print(int(rain_to_date_mm[:, -1].argmax()))   # 1: wetter after day 4, a different station

## Exercise 6: Missing data and interpolation

Given hourly air temperatures with a gap of two readings,

```python
temp_celsius = np.array([12.0, 13.5, np.nan, np.nan, 18.0, 18.5])
```

fill the gap by linear interpolation against the integer index. Then print the NaN-aware mean of the
original series and the ordinary mean of the filled one. The two differ; say in a comment why.

In [ ]:
import numpy as np

temp_celsius = np.array([12.0, 13.5, np.nan, np.nan, 18.0, 18.5])
hours = np.arange(temp_celsius.size)
present = ~np.isnan(temp_celsius)             # True where a reading exists

filled_celsius = np.interp(hours, hours[present], temp_celsius[present])
print(filled_celsius)                         # [12.  13.5 15.  16.5 18.  18.5]

print(np.nanmean(temp_celsius))               # 15.5: the four real readings
print(filled_celsius.mean())                  # 15.58...: six values, two of them filled
# np.nanmean averages only the four real readings. Filling adds two values on the straight
# line from 13.5 to 18.0; they average 15.75, above the 15.5 of the real readings, so they
# pull the mean up. np.nanmean ignores the gap, while the filled mean assumes the temperature
# changed linearly across it.

## Exercise 7: Avoid the dtype trap

An assistant wrote this function to compute each cell's share of the total rainfall in an integer
field of rain-gauge readings:

```python
def rain_share(rain_mm):
    share = np.zeros_like(rain_mm)
    share[:] = rain_mm / rain_mm.sum()
    return share

rain_mm = np.array([[0, 2, 5], [1, 0, 8], [3, 4, 2]])
```

Call it and print the result. Explain in a comment why every share is 0, then fix the function
twice: once keeping the preallocation but passing `dtype=float` to `np.zeros_like`, and once
without preallocating at all. Confirm that each fixed version returns a float array whose shares
sum to 1.

In [ ]:
import numpy as np

def rain_share(rain_mm):
    share = np.zeros_like(rain_mm)
    share[:] = rain_mm / rain_mm.sum()
    return share

rain_mm = np.array([[0, 2, 5], [1, 0, 8], [3, 4, 2]])
print(rain_share(rain_mm))
# np.zeros_like copies the integer dtype of rain_mm. Every share lies between 0 and 1, so
# assigning it into the integer array truncates it to 0.

def rain_share_float(rain_mm):
    share = np.zeros_like(rain_mm, dtype=float)   # preallocate as float explicitly
    share[:] = rain_mm / rain_mm.sum()
    return share

def rain_share_direct(rain_mm):
    return rain_mm / rain_mm.sum()                # division already returns floats

share_float = rain_share_float(rain_mm)
share_direct = rain_share_direct(rain_mm)
# the sums come out as 1.0000000000000002, so compare with np.allclose, not ==
print(share_float.dtype, np.allclose(share_float.sum(), 1.0))     # float64 True
print(share_direct.dtype, np.allclose(share_direct.sum(), 1.0))   # float64 True

## Exercise 8: Reshape and stack

1. Create the integers 0 to 11 as a 1D array, then reshape it into a 3×4 array and print both
   shapes.
2. Reshape the same data into a 4×3 array. Print it and say in a comment why the numbers appear
   in a different arrangement.
3. Use `reshape(-1)` to flatten your 3×4 array back to 1D, and confirm the shape.
4. Stack the 3×4 array on top of a row of zeros with `np.vstack`, and print the resulting shape.

In [ ]:
import numpy as np

flat = np.arange(12)
grid = flat.reshape(3, 4)
print(flat.shape, grid.shape)          # (12,) (3, 4)

other = flat.reshape(4, 3)
print(other)
# the data is stored in one flat block and read row by row, so changing the shape
# changes where each row ends, not the order of the values

print(grid.reshape(-1).shape)          # (12,) — -1 infers the length

stacked = np.vstack([grid, np.zeros(4)])
print(stacked.shape)                   # (4, 4)